In [ ]:
import torch
import clip
import models.vqvae as vqvae
from models.vqvae_sep import VQVAE_SEP
import models.t2m_trans as trans
import models.t2m_trans_uplow as trans_uplow
from tqdm import tqdm
import numpy as np
from exit.utils import visualize_2motions
import options.option_eeg2motion as option_trans
import sys 
import torch.nn.functional as F
from torch.utils import data
import torch.nn as nn
import scipy
import utils.utils_model as utils_model

import os
from einops import rearrange, repeat
from dataset import dataset_EM_train,dataset_EM_eval
from utils.eval_trans import calculate_R_precision,euclidean_distance_matrix
from utils.log_utils import setup_logger
from scipy import stats

In [ ]:
sys.argv = ['','--resume-pth','output/vq/2024-06-03-20-22-07_retrain/net_last.pth','--resume-trans','output/t2m/2024-06-04-09-29-20_trans_name_b128/net_last.pth']
args = option_trans.get_args_parser()
args.vq_dir = f'./output/vq/{args.vq_name}' #os.path.join("./dataset/KIT-ML" if args.dataname == 'kit' else "./dataset/HumanML3D", f'{args.vq_name}')
codebook_dir = f'{args.vq_dir}/codebook/'
from options.get_eval_option import get_opt
from models.evaluator_wrapper import EvaluatorModelWrapper

dataset_opt_path = 'checkpoints/kit/Comp_v6_KLD005/opt.txt' if args.dataname == 'kit' else 'checkpoints/t2m/Comp_v6_KLD005/opt.txt'

wrapper_opt = get_opt(dataset_opt_path, torch.device('cuda'))
eval_wrapper = EvaluatorModelWrapper(wrapper_opt)

In [ ]:
import random
def set_seed(seed):
    random.seed(seed)                      
    np.random.seed(seed)                   
    torch.manual_seed(seed)                
    torch.cuda.manual_seed(seed)           
    torch.cuda.manual_seed_all(seed)       
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False 

set_seed(args.seed)

In [ ]:
##### ---- CLIP ---- #####
clip_model, clip_preprocess = clip.load("ViT-B/32", device=torch.device('cuda'), jit=False)  # Must set jit=False for training
clip.model.convert_weights(clip_model)  # Actually this line is unnecessary since clip by default already on float16
clip_model.eval()
for p in clip_model.parameters():
    p.requires_grad = False

# https://github.com/openai/CLIP/issues/111
class TextCLIP(torch.nn.Module):
    def __init__(self, model) :
        super(TextCLIP, self).__init__()
        self.model = model
        
    def forward(self,text):
        with torch.no_grad():
            word_emb = self.model.token_embedding(text).type(self.model.dtype)
            word_emb = word_emb + self.model.positional_embedding.type(self.model.dtype)
            word_emb = word_emb.permute(1, 0, 2)  # NLD -> LND
            word_emb = self.model.transformer(word_emb)
            word_emb = self.model.ln_final(word_emb).permute(1, 0, 2).float()
            enctxt = self.model.encode_text(text).float()
        return enctxt, word_emb
clip_model = TextCLIP(clip_model)

def get_vqvae(args, is_upper_edit):
    if not is_upper_edit:
        return vqvae.HumanVQVAE(args, ## use args to define different parameters in different quantizers
                            args.nb_code,
                            args.code_dim,
                            args.output_emb_width,
                            args.down_t,
                            args.stride_t,
                            args.width,
                            args.depth,
                            args.dilation_growth_rate)
    else:
        return VQVAE_SEP(args, ## use args to define different parameters in different quantizers
                        args.nb_code,
                        args.code_dim,
                        args.output_emb_width,
                        args.down_t,
                        args.stride_t,
                        args.width,
                        args.depth,
                        args.dilation_growth_rate,
                        moment={'mean': torch.from_numpy(args.mean).cuda().float(), 
                            'std': torch.from_numpy(args.std).cuda().float()},
                        sep_decoder=True)

def get_maskdecoder(args, vqvae, is_upper_edit):
    tranformer = trans if not is_upper_edit else trans_uplow
    return tranformer.Text2Motion_Transformer(vqvae,
                                num_vq=args.nb_code, 
                                embed_dim=args.embed_dim_gpt, 
                                clip_dim=args.clip_dim, 
                                block_size=args.block_size, 
                                num_layers=args.num_layers, 
                                num_local_layer=args.num_local_layer, 
                                n_head=args.n_head_gpt, 
                                drop_out_rate=args.drop_out_rate, 
                                fc_rate=args.ff_rate)

class MMM(torch.nn.Module):
    def __init__(self, args=None, is_upper_edit=False):
        super().__init__()
        self.is_upper_edit = is_upper_edit


        args.dataname = args.dataset_name = 't2m'

        self.vqvae = get_vqvae(args, is_upper_edit)
        ckpt = torch.load(args.resume_pth, map_location='cpu')
        self.vqvae.load_state_dict(ckpt['net'], strict=True)
        if is_upper_edit:
            class VQVAE_WRAPPER(torch.nn.Module):
                def __init__(self, vqvae) :
                    super().__init__()
                    self.vqvae = vqvae
                    
                def forward(self, *args, **kwargs):
                    return self.vqvae(*args, **kwargs)
            self.vqvae = VQVAE_WRAPPER(self.vqvae)
        self.vqvae.eval()
        self.vqvae.cuda()

        self.maskdecoder = get_maskdecoder(args, self.vqvae, is_upper_edit)
        ckpt = torch.load(args.resume_trans, map_location='cpu')
        self.maskdecoder.load_state_dict(ckpt['trans'], strict=True)
        self.maskdecoder.eval()
        self.maskdecoder.cuda()

    def forward(self, text, lengths=-1, rand_pos=True):
        b = len(text)
        feat_clip_text = clip.tokenize(text, truncate=True).cuda()
        feat_clip_text, word_emb = clip_model(feat_clip_text)
        # feat_clip_text_null = clip.tokenize([''], truncate=True).cuda()
        # feat_clip_text_null, word_emb_null = clip_model(feat_clip_text_null)
        # index_motion = self.maskdecoder(feat_clip_text, word_emb_null, type="sample", m_length=lengths, rand_pos=rand_pos, if_test=False)
        index_motion,logits = self.maskdecoder(feat_clip_text, word_emb, type="sample", m_length=lengths, rand_pos=rand_pos, if_test=False)

        m_token_length = torch.ceil((lengths)/4).int()
        pred_pose_all = torch.zeros((b, 196, 263)).cuda()
        for k in range(b):
            pred_pose = self.vqvae(index_motion[k:k+1, :m_token_length[k]], type='decode')
            pred_pose_all[k:k+1, :int(lengths[k].item())] = pred_pose
        return pred_pose_all,index_motion,logits

    def inbetween_eval(self, base_pose, m_length, start_f, end_f, inbetween_text):
        bs, seq = base_pose.shape[:2]
        tokens = -1*torch.ones((bs, 50), dtype=torch.long).cuda()
        m_token_length = torch.ceil((m_length)/4).int()
        start_t = torch.round((start_f)/4).int()
        end_t = torch.round((end_f)/4).int()

        for k in range(bs):
            index_motion = self.vqvae(base_pose[k:k+1, :m_length[k]].cuda(), type='encode')
            tokens[k, :start_t[k]] = index_motion[0][:start_t[k]]
            tokens[k, end_t[k]:m_token_length[k]] = index_motion[0][end_t[k]:m_token_length[k]]

        text = clip.tokenize(inbetween_text, truncate=True).cuda()
        feat_clip_text, word_emb_clip = clip_model(text)

        mask_id = self.maskdecoder.num_vq + 2
        tokens[tokens==-1] = mask_id
        inpaint_index = self.maskdecoder(feat_clip_text, word_emb_clip, type="sample", m_length=m_length.cuda(), token_cond=tokens)

        pred_pose_eval = torch.zeros((bs, seq, base_pose.shape[-1])).cuda()
        for k in range(bs):
            pred_pose = self.vqvae(inpaint_index[k:k+1, :m_token_length[k]], type='decode')
            pred_pose_eval[k:k+1, :int(m_length[k].item())] = pred_pose
        return pred_pose_eval

    def long_range(self, text, lengths, num_transition_token=2, output='concat', index_motion=None):
        b = len(text)
        feat_clip_text = clip.tokenize(text, truncate=True).cuda()
        feat_clip_text, word_emb = clip_model(feat_clip_text)
        if index_motion is None:
            index_motion = self.maskdecoder(feat_clip_text, word_emb, type="sample", m_length=lengths, rand_pos=False)

        m_token_length = torch.ceil((lengths)/4).int()
        if output == 'eval':
            frame_length = m_token_length * 4
            m_token_length = m_token_length.clone()
            m_token_length = m_token_length - 2*num_transition_token
            m_token_length[[0,-1]] += num_transition_token # first and last have transition only half
        
        half_token_length = (m_token_length/2).int()
        idx_full_len = half_token_length >= 24
        half_token_length[idx_full_len] = half_token_length[idx_full_len] - 1

        mask_id = self.maskdecoder.num_vq + 2
        tokens = -1*torch.ones((b-1, 50), dtype=torch.long).cuda()
        transition_train_length = []
        
        for i in range(b-1):
            if output == 'concat':
                i_index_motion = index_motion[i]
                i1_index_motion = index_motion[i+1]
            if output == 'eval':
                if i == 0:
                    i_index_motion = index_motion[i, :m_token_length[i]]
                else:
                    i_index_motion = index_motion[i, num_transition_token:m_token_length[i] + num_transition_token]
                if i == b-1:
                    i1_index_motion = index_motion[i+1, :m_token_length[i+1]]
                else:
                    i1_index_motion = index_motion[i+1, 
                                                num_transition_token:m_token_length[i+1] + num_transition_token]
            left_end = half_token_length[i]
            right_start = left_end + num_transition_token
            end = right_start + half_token_length[i+1]

            tokens[i, :left_end] = i_index_motion[m_token_length[i]-left_end: m_token_length[i]]
            tokens[i, left_end:right_start] = mask_id
            tokens[i, right_start:end] = i1_index_motion[:half_token_length[i+1]]
            transition_train_length.append(end)
        transition_train_length = torch.tensor(transition_train_length).to(index_motion.device)
        text = clip.tokenize(text[:-1], truncate=True).cuda()
        feat_clip_text, word_emb_clip = clip_model(text)
        inpaint_index = self.maskdecoder(feat_clip_text, word_emb_clip, type="sample", m_length=transition_train_length*4, token_cond=tokens, max_steps=1)
        
        if output == 'concat':
            all_tokens = []
            for i in range(b-1):
                all_tokens.append(index_motion[i, :m_token_length[i]])
                all_tokens.append(inpaint_index[i, tokens[i] == mask_id])
            all_tokens.append(index_motion[-1, :m_token_length[-1]])
            all_tokens = torch.cat(all_tokens).unsqueeze(0)
            pred_pose = self.vqvae(all_tokens, type='decode')
            return pred_pose
        elif output == 'eval':
            all_tokens = []
            for i in range(b):
                motion_token = index_motion[i, :m_token_length[i]]
                if i == 0:
                    first_current_trans_tok = inpaint_index[i, tokens[i] == mask_id]
                    all_tokens.append(motion_token)
                    all_tokens.append(first_current_trans_tok)
                else:
                    if i < b-1:
                        first_current_trans_tok = inpaint_index[i, tokens[i] == mask_id]
                        all_tokens.append(motion_token)
                        all_tokens.append(first_current_trans_tok)
                    else:
                        all_tokens.append(motion_token)
            all_tokens = torch.cat(all_tokens)
            pred_pose_concat = self.vqvae(all_tokens.unsqueeze(0), type='decode')
            
            trans_frame = num_transition_token*4
            pred_pose = torch.zeros((b, 196, 263)).cuda()
            current_point = 0
            for i in range(b):
                if i == 0:
                    start_f = torch.tensor(0)
                    end_f = frame_length[i]
                else:
                    start_f = current_point - trans_frame
                    end_f = start_f + frame_length[i]
                current_point = end_f
                pred_pose[i, :frame_length[i]] = pred_pose_concat[0, start_f: end_f]
            return pred_pose

    def upper_edit(self, pose, m_length, upper_text, lower_mask=None):
        pose = pose.clone().cuda().float() # bs, nb_joints, joints_dim, seq_len
        m_tokens_len = torch.ceil((m_length)/4)
        bs, seq = pose.shape[:2]
        max_motion_length = int(seq/4) + 1
        mot_end_idx = self.vqvae.vqvae.num_code
        mot_pad_idx = self.vqvae.vqvae.num_code + 1
        mask_id = self.vqvae.vqvae.num_code + 2
        target_lower = []
        for k in range(bs):
            target = self.vqvae(pose[k:k+1, :m_length[k]], type='encode')
            if m_tokens_len[k]+1 < max_motion_length:
                target = torch.cat([target, 
                                    torch.ones((1, 1, 2), dtype=int, device=target.device) * mot_end_idx, 
                                    torch.ones((1, max_motion_length-1-m_tokens_len[k].int().item(), 2), dtype=int, device=target.device) * mot_pad_idx], axis=1)
            else:
                target = torch.cat([target, 
                                    torch.ones((1, 1, 2), dtype=int, device=target.device) * mot_end_idx], axis=1)
            target_lower.append(target[..., 1])
        target_lower = torch.cat(target_lower, axis=0)

        ### lower mask ###
        if lower_mask is not None:
            lower_mask = torch.cat([lower_mask, torch.zeros(bs, 1, dtype=int)], dim=1).bool()
            target_lower_masked = target_lower.clone()
            target_lower_masked[lower_mask] = mask_id
            select_end = target_lower == mot_end_idx
            target_lower_masked[select_end] = target_lower[select_end]
        else:
            target_lower_masked = target_lower
        ##################

        pred_len = m_length.cuda()
        pred_tok_len = m_tokens_len
        pred_pose_eval = torch.zeros((bs, seq, pose.shape[-1])).cuda()

        # __upper_text__ = ['A man punches with right hand.'] * 32
        text = clip.tokenize(upper_text, truncate=True).cuda()
        feat_clip_text, word_emb_clip = clip_model(text)
        # index_motion = trans_encoder(feat_clip_text, idx_lower=target_lower_masked, word_emb=word_emb_clip, type="sample", m_length=pred_len, rand_pos=True, CFG=-1)
        index_motion = self.maskdecoder(feat_clip_text, target_lower_masked, word_emb_clip, type="sample", m_length=pred_len, rand_pos=True)
        for i in range(bs):
            all_tokens = torch.cat([
                index_motion[i:i+1, :int(pred_tok_len[i].item()), None],
                target_lower[i:i+1, :int(pred_tok_len[i].item()), None]
            ], axis=-1)
            pred_pose = self.vqvae(all_tokens, type='decode')
            pred_pose_eval[i:i+1, :int(pred_len[i].item())] = pred_pose

        return pred_pose_eval

sys.argv = ['','--resume-pth','output/vq/2024-06-03-20-22-07_retrain/net_last.pth','--resume-trans','output/t2m/2024-06-04-09-29-20_trans_name_b128/net_last.pth']
args = option_trans.get_args_parser()
mmm = MMM(args).cuda()

In [ ]:
args.vq_dir = f'./output/vq/{args.vq_name}' #os.path.join("./dataset/KIT-ML" if args.dataname == 'kit' else "./dataset/HumanML3D", f'{args.vq_name}')
codebook_dir = f'{args.vq_dir}/codebook/'
from options.get_eval_option import get_opt
from models.evaluator_wrapper import EvaluatorModelWrapper

dataset_opt_path = 'checkpoints/kit/Comp_v6_KLD005/opt.txt' if args.dataname == 'kit' else 'checkpoints/t2m/Comp_v6_KLD005/opt.txt'

wrapper_opt = get_opt(dataset_opt_path, torch.device('cuda'))
eval_wrapper = EvaluatorModelWrapper(wrapper_opt)

In [ ]:
def get_motion_feat_t2m(dataset,mmm,eval_wrapper):
    all_motion_feat = []

    for i in range(len(dataset)):
        clip_text_train, train_motion, train_motion_len, eeg_train,subid,motion_key = dataset[i]

        train_motion = train_motion.long().cuda()

        true_pose = torch.zeros((1, 196, 263)).cuda()
        true_pose_ = mmm.vqvae(
            train_motion[:train_motion_len].unsqueeze(0),
            type='decode'
        )
        true_pose[:, :train_motion_len*4] = true_pose_

        movements = eval_wrapper.movement_encoder(true_pose[..., :-4]).detach()
        m_lens = torch.tensor([train_motion_len*4 // eval_wrapper.opt.unit_length]).cuda()

        motion_embedding = eval_wrapper.motion_encoder(movements, m_lens)

        motion_feat = motion_embedding  # [1, 512]

        all_motion_feat.append(motion_feat.cpu())

    all_motion_feat = torch.cat(all_motion_feat, dim=0)  # [N, 512]
    return all_motion_feat

def get_text_feat_clip(dataset,clip,clip_model):
    all_text_feat = []

    for i in range(len(dataset)):
        clip_text_train, train_motion, train_motion_len, eeg_train,subid,motion_key = dataset[i]

        feat_clip_text = clip.tokenize([clip_text_train], truncate=True).cuda()
        feat_clip_text, word_emb = clip_model(feat_clip_text)

        text_feat = feat_clip_text  # [1, 512]

        all_text_feat.append(text_feat.cpu())

    all_text_feat = torch.cat(all_text_feat, dim=0)  # [N, 512]
    return all_text_feat

def get_test_text_feat_clip(dataset,clip,clip_model):
    all_text_feat = []

    for i in range(len(dataset)):
        word_embeddings, pos_one_hots, clip_text, sent_len, pose, m_length, token, name, eeg,subid,motion_key = dataset[i]

        feat_clip_text = clip.tokenize([clip_text], truncate=True).cuda()
        feat_clip_text, word_emb = clip_model(feat_clip_text)

        text_feat = feat_clip_text  # [1, 512]

        all_text_feat.append(text_feat.cpu())

    all_text_feat = torch.cat(all_text_feat, dim=0)  # [N, 512]
    return all_text_feat

def get_test_motion_feat_t2m(dataset,eval_wrapper):
    all_motion_feat = []

    for i in range(len(dataset)):
        word_embeddings, pos_one_hots, clip_text, sent_len, pose, m_length, token, name, eeg,subid,motion_key = dataset[i]

        pose = torch.tensor(pose).unsqueeze(0).cuda()
        
        true_pose = torch.zeros((1, 196, 263)).cuda()
        true_pose[:, :pose.shape[1]] = pose
        
        movements = eval_wrapper.movement_encoder(true_pose[..., :-4]).detach()
        m_lens = torch.tensor([m_length // eval_wrapper.opt.unit_length]).cuda()
        motion_embedding = eval_wrapper.motion_encoder(movements, m_lens)

        motion_feat = motion_embedding  # [1, 512]

        all_motion_feat.append(motion_feat.cpu())

    all_motion_feat = torch.cat(all_motion_feat, dim=0)  # [N, 512]
    return all_motion_feat

@torch.no_grad()
def get_video_feat(dataset, video_feat_root='/videomae_global_features'):

    all_video_feat = []

    for i in range(len(dataset)):

        clip_text_train, train_motion, train_motion_len, eeg_train, subid, motion_key = dataset[i]

        video_key = str(motion_key)
        video_path = os.path.join(video_feat_root, f"{video_key}.npy")

        if not os.path.exists(video_path):
            print(f"[Missing npy] {video_path}")
            continue

        # 1. load precomputed feature
        video_feat = np.load(video_path)  # (D,) or (T,D)

        video_feat = torch.tensor(video_feat).float().cuda()

        video_feat = video_feat.unsqueeze(0)  # [1, D]

        all_video_feat.append(video_feat.cpu())

    all_video_feat = torch.cat(all_video_feat, dim=0)

    return all_video_feat

@torch.no_grad()
def get_test_video_feat(dataset, video_feat_root='/videomae_global_features'):

    all_video_feat = []

    for i in range(len(dataset)):

        word_embeddings, pos_one_hots, clip_text, sent_len, pose, m_length, token, name, eeg,subid,motion_key = dataset[i]

        video_key = str(motion_key)
        video_path = os.path.join(video_feat_root, f"{video_key}.npy")

        if not os.path.exists(video_path):
            print(f"[Missing npy] {video_path}")
            continue

        # 1. load precomputed feature
        video_feat = np.load(video_path)  # (D,) or (T,D)

        video_feat = torch.tensor(video_feat).float().cuda()

        video_feat = video_feat.unsqueeze(0)  # [1, D]

        all_video_feat.append(video_feat.cpu())

    all_video_feat = torch.cat(all_video_feat, dim=0)

    return all_video_feat

In [ ]:
args.eeg_data_root

In [ ]:
sub_n=len(args.eeg_data_root)
sub_n

In [ ]:
dataset = dataset_EM_train.EEG2MotionDataset(
        args.dataname,
        eeg_roots=args.eeg_data_root,
        eeg_name=args.eeg_data_name,
        tokenizer_name=codebook_dir,
        codebook_size=args.nb_code,
        unit_length=2**args.down_t,
        eeg_ch=args.eeg_ch
    )

In [ ]:
n_eeg_ch = dataset[0][3].shape[0]
n_eeg_ch

In [ ]:
args.clip_target

In [ ]:
if  args.clip_target=='motion_t2m':
    target_feats = get_motion_feat_t2m(dataset,mmm,eval_wrapper).detach()
elif args.clip_target=='text_clip':
    target_feats = get_text_feat_clip(dataset,clip,clip_model).detach()
elif args.clip_target=='video_mae':
    target_feats = get_video_feat(dataset, video_feat_root=args.video_feat_root).detach()

if args.random_level:
    print("random shuffle")
    idx = torch.randperm(target_feats.size(0))
    target_feats = target_feats[idx]

train_with_feat_loader = dataset_EM_train.EEG2MotionLoaderWithFeat(args,base_dataset=dataset,target_feats=target_feats)
train_with_feat_loader_iter = dataset_EM_train.cycle(train_with_feat_loader)

In [ ]:
val_dataset = dataset_EM_eval.EEG2MotionTestDataset(
    dataset_name=args.dataname,
    eeg_roots=args.eeg_data_root,
    eeg_name=args.eeg_data_name,
    test_avg=args.test_avg,
    dataset_type="val",
    eeg_ch=args.eeg_ch
)
if  args.clip_target=='motion_t2m':
    target_feats_val = get_test_motion_feat_t2m(val_dataset,eval_wrapper).detach()
elif args.clip_target=='text_clip':
    target_feats_val = get_test_text_feat_clip(val_dataset,clip,clip_model).detach()
elif args.clip_target=='video_mae':
    target_feats_val = get_test_video_feat(val_dataset, video_feat_root=args.video_feat_root).detach()
val_with_feat_loader = dataset_EM_eval.EEG2MotionLoaderWithFeat(base_dataset=val_dataset,target_feats=target_feats_val)
val_with_feat_loader_iter = dataset_EM_train.cycle(val_with_feat_loader)

In [ ]:
test_dataset = dataset_EM_eval.EEG2MotionTestDataset(
    dataset_name=args.dataname,
    eeg_roots=args.eeg_data_root,
    eeg_name=args.eeg_data_name,
    test_avg=args.test_avg,
    dataset_type="test",
    eeg_ch=args.eeg_ch
)
if  args.clip_target=='motion_t2m':
    target_feats_test = get_test_motion_feat_t2m(test_dataset,eval_wrapper).detach()
elif args.clip_target=='text_clip':
    target_feats_test = get_test_text_feat_clip(test_dataset,clip,clip_model).detach()
elif args.clip_target=='video_mae':
    target_feats_test = get_test_video_feat(test_dataset, video_feat_root=args.video_feat_root).detach()
test_with_feat_loader = dataset_EM_eval.EEG2MotionLoaderWithFeat(base_dataset=test_dataset,target_feats=target_feats_test)
test_with_feat_loader_iter = dataset_EM_train.cycle(test_with_feat_loader)

In [ ]:
if  args.clip_target=='motion_t2m':
    output_dim = 512 
elif args.clip_target=='text_clip':
    output_dim = 512 
elif args.clip_target=='video_mae':
    output_dim = 768

# EEG encoder

In [ ]:
from torch import Tensor
from einops.layers.torch import Rearrange, Reduce
import math
class AdaBN2d(nn.Module):
    def __init__(self, num_features: int, n_subjects: int,
                 affine: bool = True, track_running_stats: bool = True):
        super().__init__()
        self.n_subjects = n_subjects
        self.bns = nn.ModuleList([
            nn.BatchNorm2d(num_features, affine=affine,
                           track_running_stats=track_running_stats)
            for _ in range(n_subjects)
        ])

    def forward(self, x: Tensor, subid: Tensor) -> Tensor:
        assert (subid >= 0).all() and (subid < self.n_subjects).all(), \
            f"subid must be in [0, {self.n_subjects-1}]"

        out = torch.empty_like(x)  
        for i in range(self.n_subjects):
            mask = (subid == i)
            if mask.any():
                out[mask] = self.bns[i](x[mask])
        return out


class PatchEmbedding(nn.Module):
    """PatchEmbedding with AdaBN for multiple subjects"""
    def __init__(self, emb_size: int = 40, n_subjects: int = 1,ch = 59):
        super().__init__()
        self.n_subjects = n_subjects

        self.conv1 = nn.Conv2d(1, emb_size, (1, 40), (1, 5))
        self.conv2 = nn.Conv2d(emb_size, emb_size, (ch, 1), (1, 1))
        self.ada_bn = AdaBN2d(emb_size, n_subjects) 
        self.elu = nn.ELU()
        self.pool = nn.AvgPool2d((1, 15), (1, 3))
        self.dropout = nn.Dropout(0.5)

        self.projection = nn.Sequential(
            nn.Conv2d(emb_size, emb_size, (1, 1), stride=(1, 1)),
            Rearrange('b e h w -> b (h w) e')
        )

    def forward(self, x: Tensor, subid: Tensor) -> Tensor:
        if x.dim() == 2:             # (B, T) -> (B, 1, 1, T)
            x = x.unsqueeze(1).unsqueeze(2)
        elif x.dim() == 3:           # (B, C, T) -> (B, 1, C, T)
            x = x.unsqueeze(1)

        x = self.conv1(x)
        x = self.conv2(x)
        x = self.ada_bn(x, subid)    
        x = self.elu(x)
        x = self.pool(x)
        x = self.dropout(x)
        x = self.projection(x)       # (B, patches, emb_size)
        return x


class MultiHeadAttention(nn.Module):
    def __init__(self, emb_size, num_heads, dropout):
        super().__init__()
        self.emb_size = emb_size
        self.num_heads = num_heads
        self.keys = nn.Linear(emb_size, emb_size)
        self.queries = nn.Linear(emb_size, emb_size)
        self.values = nn.Linear(emb_size, emb_size)
        self.att_drop = nn.Dropout(dropout)
        self.projection = nn.Linear(emb_size, emb_size)

    def forward(self, x: Tensor, mask: Tensor = None) -> Tensor:
        queries = rearrange(self.queries(x), "b n (h d) -> b h n d", h=self.num_heads)
        keys = rearrange(self.keys(x), "b n (h d) -> b h n d", h=self.num_heads)
        values = rearrange(self.values(x), "b n (h d) -> b h n d", h=self.num_heads)
        energy = torch.einsum('bhqd, bhkd -> bhqk', queries, keys)  
        if mask is not None:
            fill_value = torch.finfo(torch.float32).min
            energy.mask_fill(~mask, fill_value)

        scaling = self.emb_size ** (1 / 2)
        att = F.softmax(energy / scaling, dim=-1)
        att = self.att_drop(att)
        out = torch.einsum('bhal, bhlv -> bhav ', att, values)
        out = rearrange(out, "b h n d -> b n (h d)")
        out = self.projection(out)
        return out


class ResidualAdd(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        res = x
        x = self.fn(x, **kwargs)
        x += res
        return x


class FeedForwardBlock(nn.Sequential):
    def __init__(self, emb_size, expansion, drop_p):
        super().__init__(
            nn.Linear(emb_size, expansion * emb_size),
            nn.GELU(),
            nn.Dropout(drop_p),
            nn.Linear(expansion * emb_size, emb_size),
        )


class GELU(nn.Module):
    def forward(self, input: Tensor) -> Tensor:
        return input*0.5*(1.0+torch.erf(input/math.sqrt(2.0)))


class TransformerEncoderBlock(nn.Sequential):
    def __init__(self,
                 emb_size,
                 num_heads=10,
                 drop_p=0.5,
                 forward_expansion=4,
                 forward_drop_p=0.5):
        super().__init__(
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                MultiHeadAttention(emb_size, num_heads, drop_p),
                nn.Dropout(drop_p)
            )),
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                FeedForwardBlock(
                    emb_size, expansion=forward_expansion, drop_p=forward_drop_p),
                nn.Dropout(drop_p)
            )
            ))

class TransformerEncoder(nn.Sequential):
    def __init__(self, depth, emb_size):
        super().__init__(*[TransformerEncoderBlock(emb_size) for _ in range(depth)])

class Conformer(nn.Module):
    def __init__(self , emb_size=40, depth=2,sub_n=1,ch = 59,output_dim=output_dim, **kwargs):
        super().__init__()
        self.patchem = PatchEmbedding(emb_size,sub_n,ch=ch)
        self.trans = TransformerEncoder(depth, emb_size)
        self.head = None
        self.output_dim = output_dim

    def forward(self, x,subid, **kwargs):
        x = self.patchem(x,subid)
        x = self.trans(x)
        # feature_g = x.mean(dim=1)
        feature_g = torch.flatten(x, start_dim=1)
        if self.head is None:
            input_dim = feature_g.shape[1]
            output_dim = self.output_dim 
            
            self.head = nn.Sequential(
                nn.Linear(input_dim, 256), 
                nn.ELU(),                
                nn.Dropout(0.5),           
                nn.Linear(256, 32),
                nn.ELU(),
                nn.Dropout(0.3),
                nn.Linear(32, output_dim)
            ).to(feature_g.device)
        # 应用 MLP 映射
        output = self.head(feature_g)
        return x,output

eegencoder = Conformer(sub_n = sub_n,ch = n_eeg_ch).cuda()

In [ ]:
batch = next(train_with_feat_loader_iter)
clip_text, m_tokens, m_tokens_len,eeg,target_feat,subid = batch
out = eegencoder(eeg.cuda(),subid)
eeg_f_dim = out[0].shape[-1]
clip_dim = out[1].shape[-1]
eeg.shape,out[0].shape,out[1].shape,eeg_f_dim,clip_dim

In [ ]:
def clip_loss_cosine(new_embeds, frozen_embeds, temperature=0.07):
    new_embeds = F.normalize(new_embeds, p=2, dim=-1)
    frozen_embeds = F.normalize(frozen_embeds, p=2, dim=-1)
    
    logits = torch.matmul(new_embeds, frozen_embeds.T) / temperature
    
    labels = torch.arange(len(logits)).to(logits.device)
    
    loss_new = F.cross_entropy(logits, labels)
    loss_frozen = F.cross_entropy(logits.T, labels)
    
    return (loss_new + loss_frozen) / 2

def contrastive_loss_dist(new_embeds, frozen_embeds, margin=10.0):
    dist = torch.cdist(new_embeds, frozen_embeds, p=2) 
    
    batch_size = new_embeds.size(0)
    mask_matched = torch.eye(batch_size, device=new_embeds.device)
    mask_mismatched = 1 - mask_matched
    
    loss_matched = (dist.pow(2) * mask_matched).sum() / batch_size
    
    loss_mismatched = (torch.clamp(margin - dist, min=0).pow(2) * mask_mismatched).sum() / (batch_size * (batch_size - 1))
    
    return loss_matched + loss_mismatched

class ContrastiveLoss(torch.nn.Module):
    def __init__(self, margin=10.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, output1, output2, label):
        euclidean_distance = F.pairwise_distance(output1, output2, keepdim=True)
        loss_contrastive = torch.mean((1-label) * torch.pow(euclidean_distance, 2) +
                                      (label) * torch.pow(torch.clamp(self.margin - euclidean_distance, min=0.0), 2))
        return loss_contrastive

def calculate_top_k_from_similarity(mat, top_k):
    size = mat.shape[0]
    gt_mat = np.expand_dims(np.arange(size), 1).repeat(size, 1)
    bool_mat = (mat == gt_mat)
    correct_vec = False
    top_k_list = []
    for i in range(top_k):
        correct_vec = (correct_vec | bool_mat[:, i])
        top_k_list.append(correct_vec[:, None])
    top_k_mat = np.concatenate(top_k_list, axis=1)
    return top_k_mat

def calculate_R_precision_cosine_with_temp(embedding1, embedding2, top_k, logit_scale=14.28, sum_all=False):
    assert embedding1.shape[0] == embedding2.shape[0]
    assert embedding1.shape[1] == embedding2.shape[1]
    
    emb1_norm = embedding1 / np.linalg.norm(embedding1, axis=1, keepdims=True)
    emb2_norm = embedding2 / np.linalg.norm(embedding2, axis=1, keepdims=True)
    
    sim_mat = np.dot(emb1_norm, emb2_norm.T) * logit_scale
    matching_score = np.trace(sim_mat)
    
    argmax = np.argsort(-sim_mat, axis=1)
    
    top_k_mat = calculate_top_k_from_similarity(argmax, top_k)
    
    if sum_all:
        return top_k_mat.sum(axis=0), matching_score
    else:
        return top_k_mat, matching_score

In [ ]:
if isinstance(args.eeg_data_root, list):
    folder_name = "_".join([os.path.basename(path) for path in args.eeg_data_root])
else:
    folder_name = os.path.basename(args.eeg_data_root)

save_path_best = f'eeg_model/{folder_name}_cliptest_temp_best.pth'
save_path_last = f'eeg_model/{folder_name}_cliptest_temp_last.pth'
save_path_best,save_path_last

In [ ]:
folder_name

In [ ]:
logger = setup_logger(log_name=folder_name+'_cliptest_temp')
logger.info("=== Model Parameters ===")
for arg, value in vars(args).items():
    logger.info(f"  {arg}: {value}")
logger.info("========================")

In [ ]:
optimizer = utils_model.initial_optim(args.decay_option, args.lr, args.weight_decay, eegencoder, args.optimizer)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=args.eeg_milestones, gamma=args.gamma)
bestTop3 = 0
for nb_iter in range(1, args.eeg_iter+1):
    # print(nb_iter)
    batch = next(train_with_feat_loader_iter)
    clip_text, m_tokens, m_tokens_len,eeg,target_feat,subid = batch
    eeg = eeg.float().cuda()
    target_feat = target_feat.cuda()
    x,out = eegencoder(eeg,subid)
    target_loss = 0
    
    if args.clip_loss_type == 'dist':
        target_loss = contrastive_loss_dist(out,target_feat)
    elif args.clip_loss_type == 'cosine':
        target_loss = clip_loss_cosine(out,target_feat,args.clip_temp)

    loss = target_loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    
    if nb_iter%args.val_per_epoch == 0:
        eegencoder.eval()
        target_temp_R = 0
        text_temp_R = 0
        nb_sample = 0
        for eval_iters in range(10):
            batch = next(val_with_feat_loader_iter)
            clip_text, m_tokens, m_tokens_len,eeg,target_feat,subid = batch
            target_feat = target_feat.cuda()
            eeg = eeg.float().cuda()
            x,out = eegencoder(eeg,subid)

            temp_R = np.array([0,0,0])
            
            if args.clip_loss_type == 'dist':
                temp_R, temp_match = calculate_R_precision(out.detach().cpu().numpy(), target_feat.detach().cpu().numpy(), top_k=3, sum_all=True)
            elif args.clip_loss_type == 'cosine':
                temp_R, temp_match = calculate_R_precision_cosine_with_temp(out.detach().cpu().numpy(), target_feat.detach().cpu().numpy(), top_k=3, sum_all=True)
            target_temp_R += temp_R
            
            nb_sample += len(clip_text)
        target_temp_R = target_temp_R/nb_sample

        top3mean = target_temp_R[2]
        if top3mean > bestTop3:
            logger.info('new best top3: %.6f' % top3mean)
            bestTop3 = top3mean
            torch.save(eegencoder.state_dict(), save_path_best)

        logger.info('Epoch: %d - Clip loss: %.6f - Top1: %.6f - Top2: %.6f - Top3: %.6f' % (
            nb_iter, 
            loss.detach().cpu().numpy(),
            target_temp_R[0],
            target_temp_R[1],
            target_temp_R[2]
        ))

In [ ]:
torch.save(eegencoder.state_dict(), save_path_last)

In [ ]:
eegencoder.load_state_dict(torch.load(save_path_best))
eegencoder.cuda()

In [ ]:

TOP_K_COUNT = 3

metrics = {
    'real': {'M_R': [], 'T_R': []},
    'shuf': {'M_R': [], 'T_R': []},
    'noise': {'M_R': [], 'T_R': []}
}

for random_e in range(100):
    tmp_res = {k: {m: np.zeros(TOP_K_COUNT) for m in ['M_R', 'T_R']} for k in ['real', 'shuf', 'noise']}
    nb_sample_e = 0
    
    for test_batch in iter(test_with_feat_loader):
        clip_text, m_tokens, m_tokens_len, eeg, target_feat, subid = test_batch
        bs = len(clip_text)
        nb_sample_e += bs
        
        with torch.no_grad():
            target_feat_np = target_feat.detach().cpu().numpy()
            eeg = eeg.float().cuda()

            _, out_g = eegencoder(eeg, subid)
            out_g_np = out_g.detach().cpu().numpy()

            shuf_idx = np.random.permutation(bs)
            target_feat_shuf = target_feat_np[shuf_idx]

            noise_eeg = torch.randn_like(eeg).cuda()
            _, out_g_noise = eegencoder(noise_eeg, subid)
            out_g_noise_np = out_g_noise.detach().cpu().numpy()

            def get_score_array(pred, target):
                if args.clip_loss_type == 'dist':
                    r, _ = calculate_R_precision(pred, target, top_k=TOP_K_COUNT, sum_all=True)
                else:
                    r, _ = calculate_R_precision_cosine_with_temp(pred, target, top_k=TOP_K_COUNT, sum_all=True)
                return np.array(r)

            tmp_res['real']['T_R'] += get_score_array(out_g_np, target_feat_np)
            
            tmp_res['shuf']['T_R'] += get_score_array(out_g_np, target_feat_shuf)
            
            tmp_res['noise']['T_R'] += get_score_array(out_g_noise_np, target_feat_np)

    for key in metrics.keys():
        metrics[key]['T_R'].append(tmp_res[key]['T_R'] / nb_sample_e)

def analyze_with_stats(real_list, baseline_list):
    real_arr = np.array(real_list)
    base_arr = np.array(baseline_list)
    
    real_mean = np.mean(real_arr, axis=0)
    base_mean = np.mean(base_arr, axis=0)
    
    real_std = np.std(real_arr, axis=0)
    base_std = np.std(base_arr, axis=0)
    
    _, p_values = stats.ttest_rel(real_arr, base_arr, axis=0)
    
    return {
        'real_m': real_mean, 'real_s': real_std,
        'base_m': base_mean, 'base_s': base_std,
        'p_vals': p_values
    }

logger.info("="*30)
logger.info("  FINAL STATISTICAL REPORT (Mean ± SD)  ")
logger.info("="*30)

for task in ['T_R']:
    for baseline_key in ['shuf', 'noise']:
        res = analyze_with_stats(metrics['real'][task], metrics[baseline_key][task])
        
        comp_name = "Shuffle" if baseline_key == 'shuf' else "Noise"
        logger.info(f"\n[Task: {task}] Comparison with {comp_name}:")
        
        for i in range(TOP_K_COUNT):
            rm, rs = res['real_m'][i], res['real_s'][i]
            bm, bs = res['base_m'][i], res['base_s'][i]
            pval = res['p_vals'][i]
            
            significance = ""
            if pval < 0.001: significance = "***"
            elif pval < 0.01: significance = "**"
            elif pval < 0.05: significance = "*"
            
            logger.info(
                f"  Top-{i+1}: "
                f"Real({rm:.4f} ± {rs:.4f}) vs "
                f"{comp_name}({bm:.4f} ± {bs:.4f}) | "
                f"p-val: {pval:.2e} {significance}"
            )


In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()